# Qwen-Image-Edit-2509 Inference on Trainium2

This notebook demonstrates how to run the Qwen-Image-Edit-2509 model on AWS Trainium2 (trn2) instances using Neuron SDK.

## Model Overview

Qwen-Image-Edit-2509 is an advanced image editing model that supports:
- Single and multi-image editing (1-3 input images)
- Enhanced identity preservation for persons and products
- Text editing capabilities
- Native ControlNet support

## Prerequisites

- AWS Trainium2 instance (trn2.48xlarge recommended)
- Neuron SDK installed
- Compiled models (run `compile.sh` first)

## Step 1: Install Dependencies

In [ ]:
# Install required packages
!pip install -r requirements.txt -q

## Step 2: Download and Compile Models

If you haven't compiled the models yet, run the compile script:

In [ ]:
# Download the model
!python neuron_qwen_image_edit/cache_hf_model.py

In [ ]:
# Compile the models (this may take a while)
!sh compile.sh

## Step 3: Load Pipeline and Compiled Models

In [ ]:
import os
import torch
import torch_neuronx
import neuronx_distributed
from PIL import Image
from diffusers import QwenImageEditPlusPipeline

from neuron_qwen_image_edit.neuron_commons import (
    InferenceTransformerWrapper,
    SimpleWrapper,
)

In [ ]:
# Configuration
CACHE_DIR = "qwen_image_edit_hf_cache_dir"
MODEL_ID = "Qwen/Qwen-Image-Edit-2509"
COMPILED_MODELS_DIR = "compiled_models"

# Load the pipeline
print("Loading pipeline...")
pipe = QwenImageEditPlusPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    local_files_only=True,
    cache_dir=CACHE_DIR
)
print("Pipeline loaded!")

In [ ]:
# Load compiled Neuron models
print("Loading compiled Neuron models...")

# Load compiled transformer
transformer_path = f"{COMPILED_MODELS_DIR}/transformer"
if os.path.exists(transformer_path):
    print(f"Loading compiled transformer...")
    transformer_wrapper = InferenceTransformerWrapper(pipe.transformer)
    transformer_wrapper.transformer = neuronx_distributed.trace.parallel_model_load(
        transformer_path
    )
    pipe.transformer = transformer_wrapper
    print("Transformer loaded!")

# Load compiled VAE decoder
decoder_path = f"{COMPILED_MODELS_DIR}/vae_decoder/model.pt"
if os.path.exists(decoder_path):
    print(f"Loading compiled VAE decoder...")
    vae_decoder_wrapper = SimpleWrapper(pipe.vae.decoder)
    vae_decoder_wrapper.model = torch_neuronx.DataParallel(
        torch.jit.load(decoder_path), [0, 1, 2, 3], False
    )
    pipe.vae.decoder = vae_decoder_wrapper
    print("VAE decoder loaded!")

# Load compiled VAE encoder (if exists)
encoder_path = f"{COMPILED_MODELS_DIR}/vae_encoder/model.pt"
if os.path.exists(encoder_path):
    print(f"Loading compiled VAE encoder...")
    vae_encoder_wrapper = SimpleWrapper(pipe.vae.encoder)
    vae_encoder_wrapper.model = torch_neuronx.DataParallel(
        torch.jit.load(encoder_path), [0, 1, 2, 3], False
    )
    pipe.vae.encoder = vae_encoder_wrapper
    print("VAE encoder loaded!")

print("All models loaded!")

## Step 4: Run Inference

### Example 1: Text-to-Image Generation

In [ ]:
# Text-to-image generation
prompt = "A beautiful sunset over mountains with a lake in the foreground, photorealistic"

inputs = {
    "prompt": prompt,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "true_cfg_scale": 4.0,
    "generator": torch.manual_seed(42),
    "num_images_per_prompt": 1,
}

# Warmup run
print("Running warmup...")
with torch.inference_mode():
    _ = pipe(**inputs)
print("Warmup complete!")

# Actual inference
import time
print("Running inference...")
start_time = time.time()
with torch.inference_mode():
    output = pipe(**inputs)
inference_time = time.time() - start_time
print(f"Inference completed in {inference_time:.2f}s")

# Display result
output.images[0]

### Example 2: Image Editing (Single Image)

In [ ]:
# Load an input image for editing
# Replace with your own image path
# input_image = Image.open("your_image.png")

# For demonstration, create a simple test image
import numpy as np
test_image = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

prompt = "Transform this image into a watercolor painting style"

inputs = {
    "image": [test_image],
    "prompt": prompt,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "true_cfg_scale": 4.0,
    "generator": torch.manual_seed(0),
    "num_images_per_prompt": 1,
}

print("Running image editing...")
with torch.inference_mode():
    output = pipe(**inputs)

# Display result
output.images[0]

### Example 3: Multi-Image Editing

In [ ]:
# Multi-image editing (combining 2 images)
# Replace with your own images
# image1 = Image.open("person.png")
# image2 = Image.open("background.png")

# For demonstration, create test images
image1 = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))
image2 = Image.fromarray(np.random.randint(0, 255, (512, 512, 3), dtype=np.uint8))

prompt = "Combine these two images: place the subject from the first image into the scene from the second image"

inputs = {
    "image": [image1, image2],
    "prompt": prompt,
    "negative_prompt": " ",
    "num_inference_steps": 40,
    "guidance_scale": 1.0,
    "true_cfg_scale": 4.0,
    "generator": torch.manual_seed(0),
    "num_images_per_prompt": 1,
}

print("Running multi-image editing...")
with torch.inference_mode():
    output = pipe(**inputs)

# Display result
output.images[0]

## Step 5: Save Results

In [ ]:
# Save the generated image
output.images[0].save("generated_image.png")
print("Image saved to generated_image.png")

## Notes

- The text encoder (Qwen2.5-VL) runs on CPU due to its complexity as a multimodal model
- The transformer and VAE are compiled for Neuron and run on Trainium2
- First inference includes warmup time; subsequent inferences will be faster
- For best performance, use batch sizes that match the compiled configuration